# **SPECTRA training notebook**

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import argparse
import yaml
import pickle
import torch
import networkx as nx
import scanpy as sc
import numpy as np
import pandas as pd
import warnings
import os

# Suppress annoying warnings for a clean console
warnings.filterwarnings("ignore")

In [3]:
# torch device setting
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(torch.version.cuda)
    print(torch.cuda.get_device_name())
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

12.1
NVIDIA A100 80GB PCIe
Using device: cuda


In [4]:
from spectra.utils import set_seed
from spectra.data import (
    data_preprocessing, 
    build_model_dataloaders_perts_split, 
    build_model_dataloaders_from_perts_list
    compute_weights, 
    compute_weights_enhanced
)
from spectra.model import SPECTRA
from spectra.training import train

### **Load config file**

In [5]:
config_path = '../configs/vcc_config_final.yaml'

with open(config_path, 'r') as file:
    config = yaml.safe_load(file)
print(yaml.dump(config, indent=2, sort_keys=False, default_flow_style=False))

project:
  name: k562
  seed: 42
  deterministic: false
  use_wandb: true
data:
  adata_path: /group/sottoriva/michele.calabro/SPECTRA/data/rep_k562/repk562_filtered_min100cells_40res_60cellres.h5ad
  network_path: /group/sottoriva/michele.calabro/SPECTRA/data/rep_k562/fungi_b10_k4_prototype/k562_prototype_1.tsv
  scgpt_embeddings_path: /group/sottoriva/michele.calabro/SPECTRA/data/vcc/scGPT_embeddings_all_genes.pkl
  gene_weights_path: /group/sottoriva/michele.calabro/SPECTRA/data/rep_k562/gene_weights-k562.pkl
  split_path: /group/sottoriva/michele.calabro/SPECTRA/data/rep_k562/fungi_b10_k4_prototype/K562_GWPS_split_indices.json
model:
  architecture: k562
  weights_folder_path: /group/sottoriva/michele.calabro/SPECTRA/weights
  conv_type: FAGCN
  residual_weight: 0.65
  n_channels: 16
  dropout_p: 0.1
  num_node_features: 1
training:
  batch_size: 128
  lr: 0.001
  n_epochs: 50
  alpha: 4.6
  beta: 0.05
  gamma: 4.15
  eta: 1.95
  test_ratio: 0.2
  val_ratio: 0.1



In [6]:
# Initialization & Seeding - set it to deterministic for reproducibility
set_seed(seed=config['project']['seed'], deterministic=config['project']['deterministic'])

### **Load files**

In [ ]:
# Load adata
adata = sc.read_h5ad(config['data']['adata_path'])
adata = data_preprocessing(adata,
    logtransform=True, 
    min_cells_per_pert=50)
adata

AnnData object with n_obs × n_vars = 1842145 × 8248
    obs: 'gem_group', 'target_gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'n_genes'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'n_cells', 'gene_name'
    uns: 'log1p'

In [8]:
# scGPT gene embeddings
with open(config['data']['scgpt_embeddings_path'], "rb") as f:
    scgpt_dict = pickle.load(f)

In [9]:
# load GRN
network_data = pd.read_csv(config['data']['network_path'], sep='\t')
G = nx.DiGraph()
for _, edge in network_data.iterrows():
    G.add_edge(edge['source'], edge['target'], weight=edge['weight'])

G.remove_nodes_from([n for n in G.nodes if n not in scgpt_dict])
grn_genes = set(G.nodes)

In [10]:
# Filter adata to match final network genes (gene_list)
gene_list = grn_genes & set(adata.var_names)

if grn_genes != gene_list:
    print('WARNING: some genes in the provided gene list are not included in the grn, or the gene embeddings are missing; filtering them out...')
adata = adata[:, adata.var_names.isin(gene_list)]
G = G.subgraph(gene_list).copy()

num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
print("Number of nodes:", num_nodes)
print("Number of edges:", num_edges)

Number of nodes: 4858
Number of edges: 139320


In [11]:
# Filter out perturbations that aren't in the gene list
perturbations = list(adata.obs['target_gene'].unique())
perturbations.remove('non-targeting')
perts_not_included = list({
    pert for pert in perturbations
    if any(single_pert not in gene_list for single_pert in pert.split('+'))
})
if len(perts_not_included)>0:
    print('WARNING: some perturbed genes are not included in the GRN. Filtering these perturbation samples out...')
adata = adata[~adata.obs['target_gene'].isin(perts_not_included)].copy()

### **Edge indices, embeddings matrix**
Other preparation steps fro SPECTRA

In [12]:
# sanity check
assert set(G.nodes) == set(adata.var_names), "Nodes in G and adata.var_names differ!"

In [13]:
# Map Edge Index
gene_to_idx = {node: i for i, node in enumerate(adata.var_names)}
edges = [(gene_to_idx[u], gene_to_idx[v]) for u, v in G.edges()]
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

In [14]:
# Build embedding matrix
scgpt_dict = {gene_to_idx[k]: v for k, v in scgpt_dict.items() if k in gene_to_idx}
scgpt_dim = len(next(iter(scgpt_dict.values())))
embedding_matrix = torch.zeros((num_nodes, scgpt_dim))
for gene_id, emb in scgpt_dict.items():
    embedding_matrix[gene_id] = torch.tensor(emb, dtype=torch.float32)

### **Dataloaders creation**

In [15]:
perturbations = sorted(pert for pert in adata.obs["target_gene"].unique() if pert != "non-targeting")

# Merge config dictionaries for the model/dataloaders
model_config = {**config['model'], **config['training']}
model_config['dataset_size'] = adata.shape[0]
model_config['pert_to_idx'] = {pert: i for i, pert in enumerate(perturbations)}

In [ ]:
train_loader, val_loader, test_loader, _, _, _, train_adata, _, _ = build_model_dataloaders_from_perts_list(
    adata, model_config, config['data']['split_path']
)

In [ ]:
# Load WMSE Weights
weights_path = config['data']['gene_weights_path']
if os.path.exists(weights_path):
    print('found already existing gene weights dictionary! Loading...')
    with open(weights_path, 'rb') as f:
        gene_weights = pickle.load(f)
else:
    print('No gene weights dictionary found. Calculating...')
    gene_weights = compute_weights_enhanced(adata, cells_per_pert=128)
    with open(weights_path, 'wb') as f:
        pickle.dump(gene_weights, f)

found already existing gene weights dictionary! Loading...


### **Model initialization**

In [ ]:
# Initialize W&B
wandb_support = config['project']['use_wandb']
if wandb_support:
    import wandb
    wandb.login()
    wandb.init(project=config['project']['name'], config=model_config)

In [ ]:
model = SPECTRA(
    edge_index=edge_index, 
    num_nodes=num_nodes, 
    device=device, 
    config=model_config,
    gene_embeddings=embedding_matrix,
    gene_names=adata.var_names,
    gene_weights=gene_weights
).to(device)
print(model)

SPECTRA(
  (gene_embeddings): Embedding(4922, 512)
  (project_gene): Linear(in_features=512, out_features=16, bias=True)
  (film_layer): GeneExpressionFiLM(
    (scale): Linear(in_features=1, out_features=16, bias=True)
    (shift): Linear(in_features=1, out_features=16, bias=True)
  )
  (ko_mlp): MLP(
    (network): Sequential(
      (0): Dropout(p=0.0, inplace=False)
      (1): Linear(in_features=512, out_features=16, bias=True)
    )
  )
  (encoder): VariationalGraphEncoder(
    (conv1): DirFAGCNConv(16)
    (conv2): DirFAGCNConv(16)
    (ln1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    (ln2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    (proj_mu): Linear(in_features=16, out_features=16, bias=True)
    (proj_logstd): Linear(in_features=16, out_features=16, bias=True)
  )
  (gex_decoder): FeatureDecoder(
    (conv1): DirFAGCNConv(16)
    (conv2): DirFAGCNConv(16)
    (conv3): DirFAGCNConv(16)
    (ln1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  

### **Training**

In [ ]:
idx_to_gene = {v: k for k, v in gene_to_idx.items()}
train(
    model=model, 
    train_loader=train_loader, 
    test_loader=val_loader,
    device=device,
    wandb_support=wandb_support,
    var_names=adata.var_names.tolist(),
    patience=7,
)

In [ ]:
# exit
if wandb_support:
    wandb.finish()
print('model trained and ready to go! Enjoy!')

In [ ]:
weights_dir = config['model']['weights_folder_path']
if wandb_support and wandb.run is not None:
    final_filepath = os.path.join(weights_dir,f"final_model_{model.architecture_name}_{wandb.run.id}.pth")
else:
    final_filepath = os.path.join(weights_dir, f"final_model_{model.architecture_name}_{uuid.uuid4().hex}.pth")
torch.save(model.state_dict(), final_filepath)